In [0]:
%run "/Workspace/Users/pjadhav564@gmail.com/Brazil_project/functions"

In [0]:
# read silver_path and write to gold_path
silver_path = "/Volumes/e_commerce_brazil/e_com_silver/updated_silver"
gold_path  =  "/Volumes/e_commerce_brazil/e_com_gold/business_ready/"

In [0]:
df_clean_orders = read_csv(f"{silver_path}/orders_silver")
df_clean_order_items = read_csv(f"{silver_path}/order_items_silver")
df_clean_payments = read_csv(f"{silver_path}/payments_silver")
df_clean_reviews = read_csv(f"{silver_path}/review_silver")

df_payment_agg = df_clean_payments.groupBy("order_id").agg(sum("payment_value").alias("total_payment"))
df_reviews_agg = df_clean_reviews.groupBy("order_id").agg(avg("review_score").alias("avg_review_score"))


fact_orders= df_clean_orders.alias("o")\
    .join(df_clean_order_items.alias("i"),"order_id","inner")\
    .join(df_payment_agg.alias("p"),"order_id","left")\
    .join(df_reviews_agg.alias("r"),"order_id","left")\
    .withColumn("date_id",to_date("order_purchase_timestamp"))\
    .withColumn("delivery_days",datediff(col("o.order_delivered_customer_date"),col("o.order_purchase_timestamp")))\
    .select(
        col("o.order_id"),
        col("i.product_id"),
        col("o.customer_id"),
        col("i.seller_id"),
        col("date_id"),
        col("o.order_status"),
        col("i.price"),
        col("i.freight_value"),
        col("p.total_payment"),
        col("r.avg_review_score"),
        col("delivery_days")

    )



display(fact_orders)


# write fact_orders
writefile(fact_orders,f"{gold_path}/fact_orders","overwrite")